In [8]:
import xarray as xr
import glob
import os

# ────────────────────────────────────────────────
# 1. Find and list files
# ────────────────────────────────────────────────
base_dir = "data/raw_data"
nc_files = sorted(glob.glob(os.path.join(base_dir, "*.nc")))

print(f"Working directory: {os.getcwd()}")
print(f"Searching in: {os.path.abspath(base_dir)}")
print(f"Found {len(nc_files)} NetCDF files:")
for f in nc_files:
    print(f"  → {os.path.basename(f)}")

if len(nc_files) == 0:
    print("ERROR: No files found. Check folder path / unzip again.")
else:
    # ────────────────────────────────────────────────
    # 2. Open each file individually
    # ────────────────────────────────────────────────
    datasets = []
    for filepath in nc_files:
        try:
            ds = xr.open_dataset(filepath, chunks={'time': 365})
            datasets.append(ds)
            print(f"Opened {os.path.basename(filepath)}  |  variables: {list(ds.data_vars)}")
        except Exception as e:
            print(f"Failed to open {os.path.basename(filepath)}: {e}")

    # ────────────────────────────────────────────────
    # 3. Merge them
    # ────────────────────────────────────────────────
    if len(datasets) > 0:
        try:
            combined = xr.merge(
                datasets,
                compat="override",   # take last value if conflicts
                join="exact"         # require identical coords/time
            )
            
            # ────────────────────────────────────────────────
            # 4. Print useful summary
            # ────────────────────────────────────────────────
            print("\n" + "═"*70)
            print("MERGE SUCCESSFUL – here's your combined dataset:")
            print(combined)
            
            print("\nKey information:")
            print("  Dimensions:", dict(combined.dims))
            print("  Time steps:", len(combined.time), 
                  f"({combined.time.min().values} → {combined.time.max().values})")
            print("  Variables:", list(combined.data_vars))
            print("  Coordinates:", list(combined.coords))
            
            print("\nGrid:")
            print(f"  Latitudes: {combined.lat.min().values:.3f} → {combined.lat.max().values:.3f}")
            print(f"  Longitudes: {combined.lon.min().values:.3f} → {combined.lon.max().values:.3f}")
            print(f"  Longitude style: {'0–360°' if combined.lon.max() > 180 else '-180–180°'}")
            
            # Manchester check
            man_lat = 53.48
            man_lon = 357.76 if combined.lon.max() > 180 else -2.24
            
            print(f"\nNearest point to Manchester ({man_lat}°N, {man_lon}°E):")
            nearest = combined.sel(lat=man_lat, lon=man_lon, method="nearest")
            print(nearest)
            
            if 'TREFMXAV_U' in combined:
                print("\nSample TREFMXAV_U values at nearest point (first + last timestep):")
                print(combined['TREFMXAV_U'].sel(lat=man_lat, lon=man_lon, method="nearest").isel(time=[0, -1]))
                
        except Exception as e:
            print("\nMERGE FAILED:", str(e))
            print("Tip: Inspect one file manually → run: print(datasets[0])")
    else:
        print("No datasets were successfully opened.")

Working directory: C:\Users\hp\OneDrive - The University of Manchester\Documents\GitHub\Earth-Env-DS-MSc-Course\EART60702_project2_group3
Searching in: C:\Users\hp\OneDrive - The University of Manchester\Documents\GitHub\Earth-Env-DS-MSc-Course\EART60702_project2_group3\data\raw_data
Found 6 NetCDF files:
  → 003_2006_2080_352_360.nc
  → 004_2006_2080_352_360.nc
  → 005_2006_2080_352_360.nc
  → 006_2006_2080_352_360.nc
  → 007_2006_2080_352_360.nc
  → 008_2006_2080_352_360.nc
Opened 003_2006_2080_352_360.nc  |  variables: ['TREFMXAV_U', 'FLNS', 'FSNS', 'PRECT', 'PRSN', 'QBOT', 'TREFHT', 'UBOT', 'VBOT']
Opened 004_2006_2080_352_360.nc  |  variables: ['TREFMXAV_U', 'FLNS', 'FSNS', 'PRECT', 'PRSN', 'QBOT', 'TREFHT', 'UBOT', 'VBOT']
Opened 005_2006_2080_352_360.nc  |  variables: ['TREFMXAV_U', 'FLNS', 'FSNS', 'PRECT', 'PRSN', 'QBOT', 'TREFHT', 'UBOT', 'VBOT']


C:\Users\hp\AppData\Local\Temp\ipykernel_28820\3703330165.py:26: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(filepath, chunks={'time': 365})
C:\Users\hp\AppData\Local\Temp\ipykernel_28820\3703330165.py:26: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(filepath, chunks={'time': 365})
C:\Users\hp\AppData\Local\Temp\ipykernel_28820\3703330165.py:26: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(filepath, chunks={'time': 365})
C:\Users\hp\AppData\Local\Temp\ipykernel_28820\3703330165.py:26: UserWarning: The specified

Opened 006_2006_2080_352_360.nc  |  variables: ['TREFMXAV_U', 'FLNS', 'FSNS', 'PRECT', 'PRSN', 'QBOT', 'TREFHT', 'UBOT', 'VBOT']
Opened 007_2006_2080_352_360.nc  |  variables: ['TREFMXAV_U', 'FLNS', 'FSNS', 'PRECT', 'PRSN', 'QBOT', 'TREFHT', 'UBOT', 'VBOT']
Opened 008_2006_2080_352_360.nc  |  variables: ['TREFMXAV_U', 'FLNS', 'FSNS', 'PRECT', 'PRSN', 'QBOT', 'TREFHT', 'UBOT', 'VBOT']

══════════════════════════════════════════════════════════════════════
MERGE SUCCESSFUL – here's your combined dataset:
<xarray.Dataset> Size: 65MB
Dimensions:     (time: 27374, lat: 11, lon: 6)
Coordinates:
  * time        (time) object 219kB 2006-01-02 00:00:00 ... 2080-12-31 00:00:00
  * lat         (lat) float32 44B 49.48 50.42 51.36 52.3 ... 57.02 57.96 58.9
  * lon         (lon) float32 24B 352.5 353.8 355.0 356.2 357.5 358.8
Data variables:
    TREFMXAV_U  (time, lat, lon) float32 7MB dask.array<chunksize=(365, 11, 6), meta=np.ndarray>
    FLNS        (time, lat, lon) float32 7MB dask.array<chunksi

C:\Users\hp\AppData\Local\Temp\ipykernel_28820\3703330165.py:51: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print("  Dimensions:", dict(combined.dims))


  Time steps: 27374 (2006-01-02 00:00:00 → 2080-12-31 00:00:00)
  Variables: ['TREFMXAV_U', 'FLNS', 'FSNS', 'PRECT', 'PRSN', 'QBOT', 'TREFHT', 'UBOT', 'VBOT']
  Coordinates: ['lat', 'lon', 'time']

Grid:
  Latitudes: 49.476 → 58.901
  Longitudes: 352.500 → 358.750
  Longitude style: 0–360°

Nearest point to Manchester (53.48°N, 357.76°E):
<xarray.Dataset> Size: 1MB
Dimensions:     (time: 27374)
Coordinates:
  * time        (time) object 219kB 2006-01-02 00:00:00 ... 2080-12-31 00:00:00
    lat         float32 4B 53.25
    lon         float32 4B 357.5
Data variables:
    TREFMXAV_U  (time) float32 109kB dask.array<chunksize=(365,), meta=np.ndarray>
    FLNS        (time) float32 109kB dask.array<chunksize=(365,), meta=np.ndarray>
    FSNS        (time) float32 109kB dask.array<chunksize=(365,), meta=np.ndarray>
    PRECT       (time) float32 109kB dask.array<chunksize=(365,), meta=np.ndarray>
    PRSN        (time) float32 109kB dask.array<chunksize=(365,), meta=np.ndarray>
    QBOT    